In [1]:
import pandas as pd

FILE = "snapshots_metadata.jsonl"
MAX_LEN = 120

def truncate(val, max_len=MAX_LEN):
    val = str(val)
    return val if len(val) <= max_len else val[:max_len] + "..."

# Read just the first N lines/rows
df_head = pd.read_json(FILE, lines=True, nrows=10)

print("Columns:")
print(df_head.columns.tolist())

print("\nFirst 10 rows (truncated):")
print(df_head.applymap(truncate))

Columns:
['created_at', 'darknet_type', 'domain_id', 'domain_url', 'file_size', 'html_file_location', 'page_id', 'path', 'path_hash', 'snapshot_hash', 'snapshot_id', 'tags', 'title', 'group_number', 'snapshot_counter', 'website_id', 'group_key', 'valid_group_snapshot_count']

First 10 rows (truncated):
            created_at darknet_type domain_id                     domain_url  \
0  2020-03-27 07:20:17        torv2    163884  http://hack3q3l3szn2aib.onion   
1  2020-01-12 00:03:02        torv2    162803  http://rhe4fap34sta5o5w.onion   
2  2020-08-23 08:40:41        torv2    162803  http://rhe4fap34sta5o5w.onion   
3  2020-08-23 08:41:13        torv2    162803  http://rhe4fap34sta5o5w.onion   
4  2020-10-06 12:02:34        torv2    162803  http://rhe4fap34sta5o5w.onion   
5  2020-01-12 00:01:34        torv2    162803  http://rhe4fap34sta5o5w.onion   
6  2021-04-08 23:11:59        torv2    161282  http://jokeli46ry6sylfq.onion   
7  2020-02-19 11:46:35        torv2    161282  http://jo

/tmp/ipykernel_688399/2150235321.py:17: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  print(df_head.applymap(truncate))


In [2]:
# =========================
# Build comparison table from saved doc_topics.csv only
# No BERTopic.load(...) needed
# =========================
from pathlib import Path
import pandas as pd

ROOTS = {
    "MiniLM": Path("./MiniLM"),
    "AttackBERT": Path("./AttackBERT"),
    "DarkBERT": Path("./DarkBERT"),
}

rows = []

for model_name, root in ROOTS.items():
    if not root.exists():
        print(f"[WARN] Missing root folder: {root}")
        continue

    for run_dir in sorted(root.glob("output_*")):
        doc_topics_path = run_dir / "doc_topics.csv"

        if not doc_topics_path.exists():
            print(f"[WARN] Missing doc_topics.csv in {run_dir}")
            continue

        df = pd.read_csv(doc_topics_path)

        if "topic" not in df.columns:
            print(f"[WARN] No 'topic' column in {doc_topics_path}")
            continue

        topics_series = df["topic"]

        outliers = int((topics_series == -1).sum())
        topic_sizes = topics_series[topics_series != -1].value_counts().sort_index()
        topic_count = int(topic_sizes.shape[0])
        min_cluster = int(topic_sizes.min()) if topic_count > 0 else 0

        parameters = run_dir.name.split("__")[-1]

        rows.append({
            "Model": model_name,
            "Parameters": parameters,
            "Topics": topic_count,
            "Outliers": outliers,
            "Min": min_cluster,
        })

summary_df = pd.DataFrame(rows).sort_values(["Parameters", "Model"]).reset_index(drop=True)
display(summary_df)

,Model,Parameters,Topics,Outliers,Min
0,AttackBERT,1200_120,1532,975868,1200
1,DarkBERT,1200_120,1596,855891,1203
2,MiniLM,1200_120,1602,884327,1200
3,AttackBERT,1500_150,1274,963499,1500
4,DarkBERT,1500_150,1407,838806,1500
5,MiniLM,1500_150,1468,937896,1504
6,AttackBERT,2000_200,820,1235456,2001
7,DarkBERT,2000_200,837,1173336,2008
8,MiniLM,2000_200,921,1141357,2000
9,AttackBERT,2500_250,659,1391916,2510


In [3]:
# =========================
# Count HTML files + dataset size
# =========================
from pathlib import Path

DATASET_DIR = Path(".")  # change this

html_count = 0
total_size = 0

for file in DATASET_DIR.rglob("*"):
    if file.is_file():
        total_size += file.stat().st_size
        if file.suffix.lower() == ".html":
            html_count += 1

size_gb = total_size / (1024**3)

print(f"HTML files: {html_count:,}")
print(f"Dataset size: {size_gb:.2f} GB")

HTML files: 11,337,512
Dataset size: 1251.11 GB


In [4]:
# =========================
# Find earliest snapshot_id
# =========================
import json
from pathlib import Path

file_path = Path("preprocessed_from_disk.jsonl")

earliest = None
latest = None

with file_path.open("r", encoding="utf-8", errors="ignore") as f:
    for line in f:
        try:
            rec = json.loads(line)
        except:
            continue

        snap = rec.get("snapshot_id")
        if not snap:
            continue

        if earliest is None or snap < earliest:
            earliest = snap

        if latest is None or snap > latest:
            latest = snap

print("Earliest snapshot:", earliest)
print("Latest snapshot:", latest)

Earliest snapshot: 3343107
Latest snapshot: 1093902017


In [6]:
import pandas as pd
from pathlib import Path

# -------------------------
# INPUT
# -------------------------
FEATURES_PATH = Path("./rq2a/recurring_vs_oneoff_topic_features_two_panel_clean_v2.csv")
OUT_PATH = Path("./rq2a/abstract_missing_info.csv")

df = pd.read_csv(FEATURES_PATH, sep=";")

required = {"Final", "Lifespan_Periods", "Recurring_Type", "Total_Count"}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"Missing columns: {missing}")

# -------------------------
# DEFINITIONS
# persistent core topics:
#   Continuous + top 25% lifespan
# transient topics:
#   bottom 25% lifespan
# -------------------------
lifespan_q75 = df["Lifespan_Periods"].quantile(0.75)
lifespan_q25 = df["Lifespan_Periods"].quantile(0.25)

df["Is_Core"] = (
    (df["Recurring_Type"] == "Continuous") &
    (df["Lifespan_Periods"] >= lifespan_q75)
)

df["Is_Transient"] = df["Lifespan_Periods"] <= lifespan_q25

total_volume = df["Total_Count"].sum()
core_volume = df.loc[df["Is_Core"], "Total_Count"].sum()
transient_volume = df.loc[df["Is_Transient"], "Total_Count"].sum()

percent_core = 100 * core_volume / total_volume if total_volume else 0
percent_transient = 100 * transient_volume / total_volume if total_volume else 0

median_lifespan_periods = df["Lifespan_Periods"].median()

summary = pd.DataFrame([{
    "Total_Topics": len(df),
    "Core_Topics_Count": int(df["Is_Core"].sum()),
    "Transient_Topics_Count": int(df["Is_Transient"].sum()),
    "Percent_Core_Topic_Volume": round(percent_core, 2),
    "Percent_Transient_Topic_Volume": round(percent_transient, 2),
    "Median_Lifespan_Periods": round(median_lifespan_periods, 2),
    "Lifespan_Q25": round(lifespan_q25, 2),
    "Lifespan_Q75": round(lifespan_q75, 2),
}])

summary.to_csv(OUT_PATH, sep=";", index=False, encoding="utf-8-sig")

print(summary.to_string(index=False))

print("\nAbstract-ready values:")
print(f"[PERCENT CORE TOPICS] = {percent_core:.2f}")
print(f"[PERCENT TRANSIENT]   = {percent_transient:.2f}")
print(f"Median lifespan       = {median_lifespan_periods:.2f} periods")

print(f"\nSaved to: {OUT_PATH.resolve()}")

 Total_Topics  Core_Topics_Count  Transient_Topics_Count  Percent_Core_Topic_Volume  Percent_Transient_Topic_Volume  Median_Lifespan_Periods  Lifespan_Q25  Lifespan_Q75
           55                 27                      18                      75.44                            2.91                     25.0          23.0          25.0

Abstract-ready values:
[PERCENT CORE TOPICS] = 75.44
[PERCENT TRANSIENT]   = 2.91
Median lifespan       = 25.00 periods

Saved to: /home/darknet/2026-01-27_201419_domain_with_snapshots/rq2a/abstract_missing_info.csv


In [5]:
from datetime import datetime

date = datetime.strptime(earliest, "%Y%m%d%H%M%S")
print(date)

TypeError: strptime() argument 1 must be str, not int

In [ ]:
import json

with open("preprocessed_from_disk.jsonl", encoding="utf-8") as f:
    rec = json.loads(next(f))

print(rec)

In [ ]:
# =========================
# Find earliest and latest snapshot
# =========================
import json
from datetime import datetime

file_path = "preprocessed_from_disk.jsonl"

earliest = None
latest = None

with open(file_path, encoding="utf-8") as f:
    for line in f:
        rec = json.loads(line)

        ts = rec.get("created_at")
        if ts is None:
            continue

        if earliest is None or ts < earliest:
            earliest = ts

        if latest is None or ts > latest:
            latest = ts

# convert milliseconds → datetime
earliest_dt = datetime.utcfromtimestamp(earliest / 1000)
latest_dt = datetime.utcfromtimestamp(latest / 1000)

print("Earliest snapshot timestamp:", earliest_dt)
print("Latest snapshot timestamp:", latest_dt)

In [ ]:
import json
from datetime import datetime

file_path = "preprocessed_from_disk.jsonl"

domains = set()
earliest = None
latest = None
docs = 0

with open(file_path, encoding="utf-8") as f:
    for line in f:
        rec = json.loads(line)

        docs += 1
        domains.add(rec["domain_url"])

        ts = rec["created_at"]

        if earliest is None or ts < earliest:
            earliest = ts

        if latest is None or ts > latest:
            latest = ts

print("Documents:", docs)
print("Unique websites:", len(domains))
print("Earliest:", datetime.utcfromtimestamp(earliest/1000))
print("Latest:", datetime.utcfromtimestamp(latest/1000))

In [ ]:
import json
from collections import defaultdict

file_path = "preprocessed_from_disk.jsonl"

domain_snapshots = defaultdict(set)

with open(file_path, encoding="utf-8") as f:
    for line in f:
        try:
            rec = json.loads(line)
        except:
            continue

        domain = rec.get("domain_url")
        snapshot = rec.get("snapshot_id")

        if domain and snapshot is not None:
            domain_snapshots[domain].add(snapshot)

# count domains with >=4 snapshots
count = sum(1 for snaps in domain_snapshots.values() if len(snaps) >= 4)

print("Websites with ≥4 snapshots:", count)

In [ ]:
# =========================
# Sample 100 websites with >=4 snapshots
# Keep max 5 random snapshots per website
# Include actual HTML code for each selected snapshot
# =========================
import json
import random
from pathlib import Path
from collections import defaultdict
from datetime import datetime
import pandas as pd

INPUT_JSONL = Path("preprocessed_from_disk.jsonl")

# Set this to the root folder that contains "htmls/"
# Example:
#   dataset/
#     preprocessed_from_disk.jsonl
#     htmls/
#         00/8c/008c6311d24df8c8e190c79e3d43874e.html
HTML_BASE_DIR = Path(".")   # <- adjust if needed

SAMPLE_SIZE_WEBSITES = 100
MIN_UNIQUE_SNAPSHOTS = 4
MAX_RANDOM_SNAPSHOTS_PER_WEBSITE = 5
RANDOM_SEED = 42

random.seed(RANDOM_SEED)

# --------------------------------------------------
# 1) Find domains with at least 4 unique snapshots
# --------------------------------------------------
domain_to_snapshot_ids = defaultdict(set)

with INPUT_JSONL.open("r", encoding="utf-8", errors="ignore") as f:
    for line in f:
        try:
            rec = json.loads(line)
        except Exception:
            continue

        domain = rec.get("domain_url")
        snapshot_id = rec.get("snapshot_id")

        if domain and snapshot_id is not None:
            domain_to_snapshot_ids[domain].add(snapshot_id)

eligible_domains = [
    domain for domain, snaps in domain_to_snapshot_ids.items()
    if len(snaps) >= MIN_UNIQUE_SNAPSHOTS
]

print(f"Eligible websites with >= {MIN_UNIQUE_SNAPSHOTS} snapshots: {len(eligible_domains):,}")

if len(eligible_domains) < SAMPLE_SIZE_WEBSITES:
    raise ValueError(
        f"Only {len(eligible_domains)} eligible websites found, "
        f"but SAMPLE_SIZE_WEBSITES={SAMPLE_SIZE_WEBSITES}"
    )

sampled_domains = sorted(random.sample(eligible_domains, SAMPLE_SIZE_WEBSITES))
sampled_domain_set = set(sampled_domains)

print(f"Sampled websites: {len(sampled_domains)}")

# --------------------------------------------------
# 2) Collect all records for sampled domains
# --------------------------------------------------
domain_rows = defaultdict(list)

with INPUT_JSONL.open("r", encoding="utf-8", errors="ignore") as f:
    for line in f:
        try:
            rec = json.loads(line)
        except Exception:
            continue

        domain = rec.get("domain_url")
        if domain not in sampled_domain_set:
            continue

        created_at = rec.get("created_at")
        created_at_readable = None
        if created_at is not None:
            try:
                created_at_readable = datetime.utcfromtimestamp(created_at / 1000).strftime("%Y-%m-%d %H:%M:%S")
            except Exception:
                pass

        domain_rows[domain].append({
            "domain_url": domain,
            "domain_id": rec.get("domain_id"),
            "snapshot_id": rec.get("snapshot_id"),
            "created_at": created_at,
            "created_at_readable": created_at_readable,
            "snapshot_hash": rec.get("snapshot_hash"),
            "page_id": rec.get("page_id"),
            "path": rec.get("path"),
            "title": rec.get("title"),
            "html_file_location": rec.get("html_file_location"),
            "file_size": rec.get("file_size"),
            "darknet_type": rec.get("darknet_type"),
        })

# --------------------------------------------------
# 3) Helper: load HTML from file
# --------------------------------------------------
def load_html_code(html_rel_path):
    if not html_rel_path:
        return None

    html_path = HTML_BASE_DIR / html_rel_path
    if not html_path.exists():
        return None

    try:
        return html_path.read_text(encoding="utf-8", errors="replace")
    except Exception:
        return None

# --------------------------------------------------
# 4) For each website:
#    choose up to 5 random snapshot_ids
#    keep one representative row per snapshot
#    add HTML code
# --------------------------------------------------
selected_rows = []
summary_rows = []

for domain in sampled_domains:
    rows = domain_rows[domain]

    snapshot_to_rows = defaultdict(list)
    for row in rows:
        snapshot_to_rows[row["snapshot_id"]].append(row)

    snapshot_ids = list(snapshot_to_rows.keys())
    chosen_snapshot_ids = random.sample(
        snapshot_ids,
        min(MAX_RANDOM_SNAPSHOTS_PER_WEBSITE, len(snapshot_ids))
    )
    chosen_snapshot_ids = sorted(chosen_snapshot_ids)

    for snap_id in chosen_snapshot_ids:
        snap_rows = snapshot_to_rows[snap_id]

        # Prefer root page, otherwise first by created_at/path
        root_candidates = [r for r in snap_rows if r.get("path") == "/"]
        if root_candidates:
            representative = root_candidates[0]
        else:
            representative = sorted(
                snap_rows,
                key=lambda r: (
                    r.get("created_at") if r.get("created_at") is not None else float("inf"),
                    str(r.get("path"))
                )
            )[0]

        html_code = load_html_code(representative.get("html_file_location"))

        selected_rows.append({
            **representative,
            "html_code": html_code,
            "html_loaded": html_code is not None,
        })

    created_values = [r["created_at"] for r in rows if r.get("created_at") is not None]
    summary_rows.append({
        "domain_url": domain,
        "domain_id": rows[0].get("domain_id"),
        "all_records_for_domain": len(rows),
        "unique_snapshots_for_domain": len(snapshot_ids),
        "selected_snapshots": len(chosen_snapshot_ids),
        "first_seen": (
            datetime.utcfromtimestamp(min(created_values) / 1000).strftime("%Y-%m-%d %H:%M:%S")
            if created_values else None
        ),
        "last_seen": (
            datetime.utcfromtimestamp(max(created_values) / 1000).strftime("%Y-%m-%d %H:%M:%S")
            if created_values else None
        ),
    })

# --------------------------------------------------
# 5) Save outputs
# --------------------------------------------------
df_selected = pd.DataFrame(selected_rows).sort_values(
    ["domain_url", "created_at", "snapshot_id"]
).reset_index(drop=True)

df_summary = pd.DataFrame(summary_rows).sort_values(
    ["unique_snapshots_for_domain", "domain_url"],
    ascending=[False, True]
).reset_index(drop=True)

out_selected_csv = Path("sample_100_websites_max5_snapshots_with_html.csv")
out_summary_csv = Path("sample_100_websites_max5_snapshots_summary.csv")

df_selected.to_csv(out_selected_csv, index=False)
df_summary.to_csv(out_summary_csv, index=False)

print(f"Saved selected snapshots with HTML to: {out_selected_csv.resolve()}")
print(f"Saved summary to: {out_summary_csv.resolve()}")

display(df_summary.head(20))
display(df_selected[[
    "domain_url", "snapshot_id", "created_at_readable",
    "title", "html_file_location", "html_loaded"
]].head(20))

In [ ]:
import json
from pathlib import Path
import pandas as pd

JSONL_PATH = Path("preprocessed_from_disk.jsonl")

DOC_FIELD_PRIMARY = "preprocessed_trafilatura"
DOC_FIELD_FALLBACK = "preprocessed_content"

total_rows = 0
total_domains = set()

filtered_rows = 0
filtered_domains = set()

domain_snapshot_counts = {}

bad_json = 0

with JSONL_PATH.open("r", encoding="utf-8", errors="ignore") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue

        total_rows += 1

        try:
            rec = json.loads(line)
        except:
            bad_json += 1
            continue

        domain = rec.get("domain_url")
        if domain:
            total_domains.add(domain)

        # -------------------------
        # SAME FILTERING AS PIPELINE
        # -------------------------
        text = rec.get(DOC_FIELD_PRIMARY, "")
        if isinstance(text, str) and text.strip():
            pass
        else:
            text = rec.get(DOC_FIELD_FALLBACK, "")
            if not (isinstance(text, str) and text.strip()):
                continue

        # valid document
        filtered_rows += 1

        if domain:
            filtered_domains.add(domain)
            domain_snapshot_counts[domain] = domain_snapshot_counts.get(domain, 0) + 1

# -------------------------
# ≥4 snapshot websites
# -------------------------
domains_min4 = [d for d, c in domain_snapshot_counts.items() if c >= 4]

# -------------------------
# PRINT RESULTS
# -------------------------
print("\n===== DATASET OVERVIEW =====\n")

print(f"Total snapshots (raw): {total_rows:,}")
print(f"Total unique websites (raw): {len(total_domains):,}")

print("\n--- After filtering (used in topic modeling) ---")
print(f"Filtered snapshots: {filtered_rows:,}")
print(f"Filtered unique websites: {len(filtered_domains):,}")

print("\n--- Stability subset (≥4 snapshots) ---")
print(f"Websites with ≥4 snapshots: {len(domains_min4):,}")

# optional dataframe for saving
overview_df = pd.DataFrame([{
    "Raw Snapshots": total_rows,
    "Raw Websites": len(total_domains),
    "Filtered Snapshots": filtered_rows,
    "Filtered Websites": len(filtered_domains),
    "Websites ≥4 Snapshots": len(domains_min4),
}])

OUT_PATH = Path("./rq1b/dataset_overview.csv")
OUT_PATH.parent.mkdir(exist_ok=True)

overview_df.to_csv(OUT_PATH, index=False)

print("\nSaved overview to:", OUT_PATH.resolve())

overview_df